In [ ]:
# look at emp_title before grouping
# Normalize employment titles and assess their diversity before occupational grouping.
import pandas as pd, numpy as np
df = pd.read_parquet("../data/clean_v2.parquet")
fit = df[df.issue_d <= pd.Timestamp("2014-12-31")].copy()
t = fit.emp_title.dropna().str.lower().str.replace(r"[^a-z ]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
print("raw distinct:", fit.emp_title.nunique(), "| normalized distinct:", t.nunique())
top = t.value_counts().head(25); print(top); print("share of titled loans covered by top 25:", round(top.sum() / len(t), 3))

In [ ]:
# group the titles by keyword
RULES = [
 ("healthcare", r"nurse|\brn\b|\br n\b|physician|doctor|medical|dental|pharmac|therap|hospital|health|clinic|paramedic"),
 ("education", r"teacher|professor|school|instructor|educat|teaching|lecturer|college|university"),
 ("public_safety_gov", r"police|firefight|fire dep|sheriff|deputy|correction|army|navy|military|marine|government|city of|county|state of|postal|usps|federal|trooper"),
 ("executive_owner", r"owner|president|\bceo\b|\bcfo\b|\bcoo\b|director|\bvp\b|partner|executive|chief|principal"),
 ("manager_supervisor", r"manager|supervisor|foreman|superintendent|\blead\b"),
 ("professional_technical", r"engineer|analyst|developer|programmer|software|\bit\b|network|consultant|accountant|attorney|lawyer|architect|scientist|technician|designer|auditor|controller"),
 ("transport_trades", r"driver|truck|pilot|delivery|mechanic|electrician|operator|machinist|welder|carpenter|plumber|construction|installer|maintenance|warehouse|assembler"),
 ("sales_customer", r"sales|account exec|representative|\brep\b|agent|broker|customer|cashier|retail|clerk|associate|teller|banker"),
 ("admin_support", r"assistant|secretary|admin|coordinator|specialist|receptionist|processor"),
 ("service_hospitality", r"server|cook|chef|bartender|waiter|housekeep|security|janitor|custodian|barber|stylist|hotel|restaurant|cleaner"),
]
def job_group(title):
    t = title.str.lower().str.replace(r"[^a-z ]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    g = np.select([t.fillna("").str.contains(p, regex=True) for _, p in RULES], [n for n, _ in RULES], default="other")
    return pd.Series(np.where(t.isna(), "unknown", g), index=title.index), t
fit["job_group"], t = job_group(fit.emp_title)
g = fit.groupby("job_group").defaulted.agg(["count", "mean"]); g["share"] = g["count"] / len(fit)
print(g.sort_values("count", ascending=False).round(3)); print(t[fit.job_group == "other"].value_counts().head(15))

In [ ]:
# extend the rules and enforce the minimum group size
RULES = [
 ("healthcare", r"nurse|\brn\b|\br n\b|\blpn\b|\bcna\b|physician|doctor|medical|dental|pharmac|therap|hospital|health|clinic|paramedic|social worker|counselor|caregiver"),
 ("education", r"teacher|professor|school|instructor|educat|teaching|lecturer|college|university"),
 ("public_safety_gov", r"police|firefight|fire dep|sheriff|deputy|correction|army|navy|military|marine|government|city of|county|state of|postal|usps|federal|trooper|letter carrier"),
 ("executive_owner", r"owner|president|\bceo\b|\bcfo\b|\bcoo\b|director|\bvp\b|partner|executive|chief|principal"),
 ("manager_supervisor", r"manager|management|supervisor|foreman|superintendent|\blead\b|team leader"),
 ("professional_technical", r"engineer|analyst|developer|programmer|software|\bit\b|network|consultant|accountant|bookkeep|attorney|lawyer|paralegal|underwriter|financial|advisor|architect|scientist|technician|\btech\b|designer|auditor|controller"),
 ("transport_trades", r"driver|truck|pilot|delivery|mechanic|electrician|operator|machinist|welder|carpenter|plumber|construction|installer|maintenance|warehouse|assembler"),
 ("sales_customer", r"sales|account exec|representative|\brep\b|agent|broker|customer|cashier|retail|clerk|associate|teller|banker"),
 ("admin_support", r"assistant|secretary|admin|coordinator|specialist|receptionist|processor"),
 ("service_hospitality", r"server|cook|chef|bartender|waiter|housekeep|security|janitor|custodian|barber|stylist|hotel|restaurant|cleaner|attendant"),
]
MIN_GROUP = 300
def job_group(title, small=()):
    t = title.str.lower().str.replace(r"[^a-z ]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    g = np.select([t.fillna("").str.contains(p, regex=True) for _, p in RULES], [n for n, _ in RULES], default="other")
    g = pd.Series(np.where(t.isna(), "unknown", g), index=title.index)
    return g.where(~g.isin(small), "other"), t
fit["job_group"], t = job_group(fit.emp_title)
cnt = fit.job_group.value_counts()
small = [k for k in cnt.index if cnt[k] < MIN_GROUP and k not in ("other", "unknown")]
fit["job_group"], t = job_group(fit.emp_title, small)
print("merged into other:", small)
g = fit.groupby("job_group").defaulted.agg(["count", "mean"]); g["share"] = g["count"] / len(fit)
print(g.sort_values("count", ascending=False).round(3)); print(t[fit.job_group == "other"].value_counts().head(10))

In [ ]:
# group the remaining categoricals , Normalize employment titles and assess their diversity before occupational grouping.
print(sorted(fit.emp_length.dropna().unique()))
emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}
fit["emp_length_num"] = fit.emp_length.map(emp_map)
print("unmapped non-missing values:", (fit.emp_length.notna() & fit.emp_length_num.isna()).sum())
def rare(s, label):
    c = s.value_counts(); keep = c[c >= MIN_GROUP].index
    return s.where(s.isin(keep), label), sorted(set(c.index) - set(keep))
fit["purpose_g"], rp = rare(fit.purpose, "other_purpose"); fit["state_g"], rs = rare(fit.addr_state, "OTHER_STATE")
print("purpose merged:", rp); print("states merged:", len(rs), rs)
for c in ["emp_length_num", "purpose_g", "home_ownership"]:
    print(fit.groupby(c, dropna=False).defaulted.agg(["count", "mean"]).round(3))
print(fit.groupby("state_g").defaulted.agg(["count", "mean"]).round(3).sort_values("count", ascending=False).head(12))

In [ ]:
# Weight of Evidence and Information Value
import json, sys; sys.path.append("..")
from src.features import add_features
spec = json.load(open("../reports/caps.json"))
fit = fit.join(add_features(fit, spec["features"]["credit_history_months_fill"], spec["annual_inc"]["cap_p99.5"]))
fit["emp_length_g"] = pd.cut(fit.emp_length_num, [-1, 0, 9, 10], labels=["<1y", "1-9y", "10+y"]).astype(object)
fit["purpose_g"] = fit.purpose_g.replace("other_purpose", "other")
for c, k in {"inq_last_6mths": 3, "delinq_2yrs": 2, "pub_rec": 1, "pub_rec_bankruptcies": 1, "mort_acc": 3}.items():
    fit[c + "_g"] = fit[c].clip(upper=k).astype(int).astype(str)

def woe_table(x, y):
    c = pd.crosstab(x, y).reindex(columns=[0, 1], fill_value=0)
    g = (c[0] + .5) / (c[0].sum() + .5 * len(c)); b = (c[1] + .5) / (c[1].sum() + .5 * len(c)); w = np.log(g / b)
    return pd.DataFrame({"n": c.sum(axis=1), "default_rate": c[1] / c.sum(axis=1), "woe": w, "iv": (g - b) * w})

cat = ["job_group", "emp_length_g", "purpose_g", "home_ownership", "verification_status", "term", "application_type", "state_g", "grade", "sub_grade",
       "inq_last_6mths_g", "delinq_2yrs_g", "pub_rec_g", "pub_rec_bankruptcies_g", "mort_acc_g", "never_delinquent", "bc_util_missing", "bureau_recovery_flag"]
num = ["annual_inc", "loan_amnt", "installment", "int_rate", "dti", "revol_util", "bc_util", "revol_bal", "tot_cur_bal", "avg_cur_bal", "total_rev_hi_lim",
       "tot_hi_cred_lim", "open_acc", "total_acc", "acc_open_past_24mths", "num_tl_op_past_12m", "mths_since_last_delinq", "fico_mid",
       "credit_history_months", "loan_to_income", "installment_to_income"]
woe, bins = {}, {}
for c in cat:
    woe[c] = woe_table(fit[c].astype(object).where(fit[c].notna(), "MISSING").astype(str), fit.defaulted)
for c in num:
    b, edges = pd.qcut(fit[c], 10, retbins=True, duplicates="drop"); bins[c] = [float(e) for e in edges[1:-1]]
    woe[c] = woe_table(b.cat.codes.map(lambda k: "MISSING" if k < 0 else f"{k:02d}"), fit.defaulted)
iv = pd.Series({c: t.iv.sum() for c, t in woe.items()}).sort_values(ascending=False)
rank = pd.DataFrame({"IV": iv, "bins": {c: len(woe[c]) for c in iv.index}, "min_n": {c: int(woe[c].n.min()) for c in iv.index}})
rank["read"] = pd.cut(rank.IV, [-1, .02, .1, .3, .5, 99], labels=["useless", "weak", "medium", "strong", "suspicious"])
print(rank.round(3))
woe_spec = {"min_group": MIN_GROUP, "job_rules": RULES, "job_merged": small, "purpose_merged": rp, "states_merged": rs, "emp_length_map": emp_map,
            "numeric_bin_edges": bins, "woe": {c: t.woe.round(6).to_dict() for c, t in woe.items()}}
json.dump(woe_spec, open("../reports/woe_spec.json", "w"), indent=1, default=str)


In [ ]:
# the full ranking, the small bins, and the new category
print(rank.round(3).to_string())
tiny = [(c, str(k), int(r.n)) for c, t in woe.items() for k, r in t.iterrows() if r.n < MIN_GROUP]
print(len(tiny), "bins below", MIN_GROUP); print(tiny)
late = df[df.issue_d > pd.Timestamp("2014-12-31")]
print(late.groupby("application_type").defaulted.agg(["count", "mean"]).round(3))

In [ ]:
# Identify weak features and undersized bins to guide feature selection and consolidation.
print(rank.iloc[24:].round(3).to_string())
tiny = [(c, str(k), int(r.n)) for c, t in woe.items() for k, r in t.iterrows() if r.n < MIN_GROUP]
print(len(tiny), "bins below", MIN_GROUP)
for c in sorted({t[0] for t in tiny}): print(c, [(k, n) for cc, k, n in tiny if cc == c])
print(pd.read_csv("../data/holdout_applications.csv", usecols=["application_type"]).application_type.value_counts())

In [ ]:
# finalize the WoE specification,# Finalize WoE bins, select reliable scorecard predictors, and save the feature specification.
for c in ["revol_util", "bc_util"]:
    x = fit[c].fillna(fit[c].median())
    b, edges = pd.qcut(x, 10, retbins=True, duplicates="drop"); bins[c] = [float(e) for e in edges[1:-1]]
    woe[c] = woe_table(b.cat.codes.map(lambda k: f"{k:02d}"), fit.defaulted)
fit["grade_g"] = fit.grade.replace({"F": "F-G", "G": "F-G"})
woe["grade_g"] = woe_table(fit.grade_g, fit.defaulted); woe.pop("grade", None); woe.pop("sub_grade", None)

iv = pd.Series({c: t.iv.sum() for c, t in woe.items()}); platform = ["grade_g", "int_rate"]
keep = [c for c in iv.index if iv[c] >= 0.02 and c not in platform and c != "bureau_recovery_flag"]
print(len(keep), "kept:", sorted(keep, key=lambda c: -iv[c]))
print("dropped (IV < 0.02):", sorted(set(iv.index) - set(keep) - set(platform) - {"bureau_recovery_flag"}))
print("smallest bin among kept:", min(int(woe[c].n.min()) for c in keep))
woe_spec.update({"numeric_bin_edges": bins, "woe": {c: t.woe.round(6).to_dict() for c, t in woe.items()}, "scorecard_vars": keep, "platform_vars": platform,
                 "excluded": {"leak": ["bureau_recovery_flag"], "constant_in_training": ["application_type"]}, "iv": iv.round(4).to_dict()})
json.dump(woe_spec, open("../reports/woe_spec.json", "w"), indent=1, default=str)